In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv")

In [5]:
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


### Data preparation

In [6]:
# Check for data types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [7]:
# Check the number of rows in the dataset
len(df)

5000

In [8]:
# Check the unique values in each column
for col in df.columns:
    print(f"\n{'='*50}")
    print(f"Column: {col}")
    print(f"Unique values: {df[col].nunique()}")
    print(f"Sample values: {df[col].unique()[:20]}")


Column: lead_source
Unique values: 5
Sample values: <StringArray>
['organic_search', 'social_media', 'referral', 'paid_ads', nan, 'events']
Length: 6, dtype: str

Column: industry
Unique values: 6
Sample values: <StringArray>
[   'technology',        'retail', 'manufacturing',     'education',
    'healthcare',             nan,       'finance']
Length: 7, dtype: str

Column: employment_status
Unique values: 4
Sample values: <StringArray>
['employed', 'student', nan, 'unemployed', 'self_employed']
Length: 5, dtype: str

Column: location
Unique values: 5
Sample values: <StringArray>
['europe', 'south_america', nan, 'north_america', 'africa', 'asia']
Length: 6, dtype: str

Column: annual_income
Unique values: 4313
Sample values: [88160. 72688. 44697.    nan 24062. 43841. 58738. 95059. 62565. 12000.
 43935. 80773. 47115. 62439. 59239. 37431. 83331. 49038. 50544. 31638.]

Column: number_of_courses_viewed
Unique values: 7
Sample values: [3 4 2 1 0 5 6]

Column: interaction_count
Unique valu

In [9]:
# Check for missing values in the dataset
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [ ]:
# Fill missing values for string columns with 'NA'

# Identify categorical and numerical columns
categorical_cols = df.select_dtypes(include=["object", "string"]).columns

numerical_cols = df.select_dtypes(
    include="number"
).columns

# Fill missing values
df[categorical_cols] = df[categorical_cols].fillna("NA")
df[numerical_cols] = df[numerical_cols].fillna(0.0)

# Confirm there are no missing values
print(df.isnull().sum().sum())

0


In [53]:
categorical_cols

Index(['lead_source', 'industry', 'employment_status', 'location'], dtype='str')

### Question 1

In [13]:
# What is the most frequent observation (mode) for the column `industry`?
df['industry'].mode()[0]

'technology'

### Question 2

In [49]:
numerical_features = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

correlation_matrix = df[numerical_features].corr()

pairs = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count")
]

for feature1, feature2 in pairs:
    corr = correlation_matrix.loc[feature1, feature2]
    print(feature1, "and", feature2, ":", round(corr, 4))

interaction_count and lead_score : 0.9157
number_of_courses_viewed and lead_score : 0.7572
number_of_courses_viewed and interaction_count : 0.7216
annual_income and interaction_count : 0.1228


### Split the data

In [16]:
from sklearn.model_selection import train_test_split

In [25]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

In [27]:
# Getting the target variable and features for training
y_train = df_train['converted'].values
X_train = df_train.drop(columns=['converted'])

# Getting the target variable and features for testing
y_test = df_test['converted'].values
X_test = df_test.drop(columns=['converted'])

# Getting the target variable and features for validation
y_val = df_val['converted'].values
X_val = df_val.drop(columns=['converted'])

### Question 3

In [19]:
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import OrdinalEncoder

In [54]:
X_train_categorical = X_train[categorical_cols]

encoder = OrdinalEncoder()
X_train_encoded = encoder.fit_transform(X_train_categorical)

mi_scores = mutual_info_classif(X_train_encoded, y_train, discrete_features=True, random_state=42)

feature_names = encoder.get_feature_names_out(categorical_cols)

mi_results = pd.Series(
    mi_scores,
    index=feature_names
).sort_values(ascending=False)

print(mi_results.round(2))

lead_source          0.03
employment_status    0.02
industry             0.00
location             0.00
dtype: float64


### Question 4

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [56]:
def logistic_regression_model(X_train, y_train, X_val, y_val, categorical_features, C=1.0):
    categorical_features = [
        col for col in categorical_features
        if col in X_train.columns
    ]

    # One-hot encode the categorical features
    X_train_encoded = pd.get_dummies(X_train, columns=categorical_features, drop_first=True)
    
    # Train the logistic regression model
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train_encoded, y_train)
    
    # One-hot encode the validation set
    X_val_encoded = pd.get_dummies(X_val, columns=categorical_features, drop_first=True)
    
    # Ensure that the validation set has the same columns as the training set
    X_val_encoded = X_val_encoded.reindex(columns=X_train_encoded.columns, fill_value=0)
    
    # Make predictions on the validation set
    val_pred = model.predict(X_val_encoded)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_val, val_pred)
    
    return accuracy

In [57]:
base_model_accuracy = logistic_regression_model(X_train, y_train, X_val, y_val, categorical_cols)
print(f"Validation accuracy: {base_model_accuracy:.2f}")

Validation accuracy: 0.65


### Question 5

In [59]:
features_to_drop = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count"
]

for feature in features_to_drop:
    X_train_reduced = X_train.drop(columns=[feature])
    X_val_reduced = X_val.drop(columns=[feature])

    accuracy_without_feature = logistic_regression_model(
        X_train_reduced,
        y_train,
        X_val_reduced,
        y_val,
        categorical_cols
    )

    difference = base_model_accuracy - accuracy_without_feature

    print(f"Feature removed: {feature}")
    print(f"Accuracy without feature: {accuracy_without_feature:.6f}")
    print(f"Accuracy difference: {difference:.6f}\n")

Feature removed: lead_source
Accuracy without feature: 0.642000
Accuracy difference: 0.003000

Feature removed: number_of_courses_viewed
Accuracy without feature: 0.643000
Accuracy difference: 0.002000

Feature removed: interaction_count
Accuracy without feature: 0.601000
Accuracy difference: 0.044000



### Question 6

In [61]:
C = [0.000001, 0.00001, 0.0001, 0.001]
for c in C:
    model_accuracy = logistic_regression_model(X_train, y_train, X_val, y_val, categorical_cols, C=c)
    print(f"Validation accuracy for C={c}: {model_accuracy:.3f}")

Validation accuracy for C=1e-06: 0.598
Validation accuracy for C=1e-05: 0.598
Validation accuracy for C=0.0001: 0.613
Validation accuracy for C=0.001: 0.645
